# Gold · Taxi, Zone and hourly Weather marts
Run after the three Silver views are populated. Validate grains and joins before publishing three rerunnable Gold views. Views follow the selected Silver window.

In [ ]:
import re
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "nyc_mobility", "Catalog")
dbutils.widgets.text("silver_schema", "nyc_silver", "Silver schema")
dbutils.widgets.text("gold_schema", "nyc_gold", "Gold schema")
catalog = dbutils.widgets.get("catalog").strip()
silver_schema = dbutils.widgets.get("silver_schema").strip()
gold_schema = dbutils.widgets.get("gold_schema").strip()
for identifier in (catalog, silver_schema, gold_schema):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier):
        raise ValueError(f"Invalid catalog/schema: {identifier!r}")

taxi_view = f"{catalog}.{silver_schema}.vw_green_taxi_clean"
zones_view = f"{catalog}.{silver_schema}.vw_taxi_zones_clean"
weather_view = f"{catalog}.{silver_schema}.vw_weather_hourly_clean"
dim_zone = f"{catalog}.{gold_schema}.dim_zone"
dim_weather = f"{catalog}.{gold_schema}.dim_weather_hour"
fact_taxi = f"{catalog}.{gold_schema}.fact_taxi_trip"

taxi = spark.table(taxi_view)
zones = spark.table(zones_view)
weather = spark.table(weather_view)
expected_taxi = ["pickup_datetime", "dropoff_datetime", "pickup_zone_id",
                 "dropoff_zone_id", "passenger_count", "trip_distance",
                 "fare_amount", "tip_amount", "total_amount", "is_zero_distance"]
expected_zones = ["location_id", "borough", "zone", "service_zone"]
expected_weather = ["observation_hour", "temperature_2m", "precipitation",
                    "wind_speed_10m", "timezone", "latitude", "longitude"]
for frame, required in ((taxi, expected_taxi), (zones, expected_zones),
                        (weather, expected_weather)):
    missing = set(required) - set(frame.columns)
    if missing:
        raise ValueError(f"Silver source lacks required columns: {sorted(missing)}")
print(f"Silver sources: taxi={taxi_view}, zones={zones_view}, weather={weather_view}")

In [ ]:
# Check the intended grain and all relationships before creating Gold views.
taxi_rows = taxi.count()
if taxi_rows == 0 or taxi.select(*expected_taxi).distinct().count() != taxi_rows:
    raise ValueError("Taxi Silver rows are empty or have duplicate projected trip values")
if zones.count() == 0 or zones.filter("location_id IS NULL").limit(1).count() or zones.select("location_id").distinct().count() != zones.count():
    raise ValueError("Zone Silver location_id must be non-null and unique")
if weather.count() == 0 or weather.filter("observation_hour IS NULL").limit(1).count() or weather.select("observation_hour").distinct().count() != weather.count():
    raise ValueError("Weather Silver observation_hour must be non-null and unique")

hour_key = F.expr("CAST(DATE_FORMAT(pickup_datetime, 'yyyy-MM-dd HH:00:00') AS TIMESTAMP_NTZ)")
relationships = (
    taxi.withColumn("weather_hour", hour_key)
    .join(zones.select(F.col("location_id").alias("pickup_match")),
          F.col("pickup_zone_id") == F.col("pickup_match"), "left")
    .join(zones.select(F.col("location_id").alias("dropoff_match")),
          F.col("dropoff_zone_id") == F.col("dropoff_match"), "left")
    .join(weather.select(F.col("observation_hour").alias("weather_match")),
          F.col("weather_hour") == F.col("weather_match"), "left")
    .agg(F.count("*").alias("joined_rows"),
         F.sum(F.col("pickup_match").isNull().cast("long")).alias("missing_pickup_zone"),
         F.sum(F.col("dropoff_match").isNull().cast("long")).alias("missing_dropoff_zone"),
         F.sum(F.col("weather_match").isNull().cast("long")).alias("missing_weather_hour"))
    .first()
)
if relationships.joined_rows != taxi_rows or any(relationships[name] for name in
    ("missing_pickup_zone", "missing_dropoff_zone", "missing_weather_hour")):
    raise ValueError(f"Gold relationship checks failed: {relationships.asDict()}")
print(f"Validated {taxi_rows} Taxi trips with unique Zone and Weather keys")

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {dim_zone} AS
    SELECT location_id, borough, zone, service_zone FROM {zones_view}
""")
spark.sql(f"""
    CREATE OR REPLACE VIEW {dim_weather} AS
    SELECT observation_hour, temperature_2m, precipitation,
           wind_speed_10m, timezone, latitude, longitude
    FROM {weather_view}
""")
# A deterministic fingerprint is sufficient for today's unique Silver projection.
# If future legitimate trips share every selected value, ingest a source row ID.
trip_values = ", ".join(expected_taxi)
spark.sql(f"""
    CREATE OR REPLACE VIEW {fact_taxi} AS
    SELECT SHA2(TO_JSON(STRUCT({trip_values})), 256) AS trip_key,
           pickup_datetime, dropoff_datetime,
           pickup_zone_id, dropoff_zone_id,
           CAST(DATE_FORMAT(pickup_datetime, 'yyyy-MM-dd HH:00:00') AS TIMESTAMP_NTZ) AS weather_hour,
           passenger_count, trip_distance, fare_amount, tip_amount,
           total_amount, is_zero_distance
    FROM {taxi_view}
""")
display(spark.sql(f"""
    SELECT COUNT(*) AS gold_trip_rows,
           COUNT(DISTINCT trip_key) AS unique_trip_keys,
           COUNT_IF(is_zero_distance) AS zero_distance_rows
    FROM {fact_taxi}
"""))
display(spark.sql(f"SELECT COUNT(*) AS zone_rows FROM {dim_zone}"))
display(spark.sql(f"SELECT COUNT(*) AS weather_hours FROM {dim_weather}"))